# Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
In this notebook, we demonstrate how vLLM-Hook enables **Activation Steering** for controlled generation.

**Paper**: [Improving Instruction-Following in Language Models through Activation Steering](https://arxiv.org/abs/2410.12877).<br />
**Authors**: Alessandro Stolfo, Vidhisha Balachandran, Safoora Yousefi, Eric Horvitz, Besmira Nushi <br />
**"TL;DR"**: Activation steering allows you to bias the model's behavior by nudging internal activations in specific directions. In this paper, authors focus on instruction following capability and compute the steering vectors as the difference in activations between inputs with and without instructions.

### Installation

If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [1]:

from pathlib import Path
import importlib
import importlib.util
import os
import re
import shutil
import site
import subprocess
import sys

REPO_URL = "https://github.com/IBM/vLLM-Hook.git"
REPO_BRANCH = "main"
REPO_NAME = "vLLM-Hook"
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.11,<0.19")
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128")

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()


def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("Running:", " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        tail = tail[-120:]
    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )


def run_capture(cmd):
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)


def norm(name):
    return name.lower().replace("_", "-")


def package_from_req_line(line: str) -> str:
    stripped = line.strip()
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())


def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    try:
        origin_url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return origin_url == expected_remote


def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None


def assert_cuda_runtime():
    try:
        import torch
    except Exception:
        torch = None
    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None
    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "This notebook requires a Colab GPU runtime with CUDA available. "
            "In Colab, use Runtime > Change runtime type > T4 GPU (or another GPU), then rerun from a fresh runtime."
        )


def subprocess_smoke_check():
    code = r'''
    import importlib.metadata as md
    import inspect

    print("torch", md.version("torch"))
    print("vllm", md.version("vllm"))
    print("vllm-hook-plugins", md.version("vllm-hook-plugins"))

    import torch
    import vllm
    from vllm.engine.arg_utils import EngineArgs
    import vllm_hook_plugins

    fields = getattr(EngineArgs, "__dataclass_fields__", {})
    params = inspect.signature(EngineArgs).parameters
    if "worker_extension_cls" not in fields and "worker_extension_cls" not in params:
        raise RuntimeError("Installed vLLM does not support worker_extension_cls")

    print("smoke check OK")
    '''
    run([sys.executable, "-c", code])


if IN_COLAB:
    expected_remote = REPO_URL.removesuffix(".git")
    existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Colab detected. Reusing existing repo at {REPO_ROOT}")
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Colab detected. Cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_ROOT} ...")
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch under {REPO_ROOT}; replacing clone with {expected_remote}")
            shutil.rmtree(REPO_ROOT)
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        else:
            print(f"Colab detected. Reusing existing clone at {REPO_ROOT}")
            print("Refreshing existing clone ...")
            run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])
    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Changed working directory to {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
FILTERED_REQ_FILE = Path("/tmp/vllm_hook_colab_requirements.txt")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_binary_deps_restarted")

print("Colab      :", IN_COLAB)
print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Repo branch :", REPO_BRANCH)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if IN_COLAB:
    assert_cuda_runtime()

if not PKG_DIR.exists():
    raise FileNotFoundError(f"Plugin directory not found: {PKG_DIR}")

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("Colab was detected but git is unavailable in the runtime.")

if REQ_FILE.exists():
    keep = []
    blocked = {"vllm", "torch", "torchvision", "torchaudio", "numpy", "scipy", "protobuf"}
    for line in REQ_FILE.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue
        package = package_from_req_line(stripped)
        if package in blocked:
            print("Skipping requirement managed by Colab install cell:", line)
            continue
        keep.append(line)
    FILTERED_REQ_FILE.write_text("\n".join(keep) + "\n")
    run([sys.executable, "-m", "pip", "install", "-r", str(FILTERED_REQ_FILE)])
else:
    print("Warning: requirement.txt not found; skipping dependency install.")

run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])

if COLAB_INSTALL_VLLM:
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    # Colab currently does not support CUDA 13 on all GPU runtimes, so use CUDA 12.x wheels.
    run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch", "torchvision", "torchaudio"])
    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        for leftover in [
            site_path / "vllm",
            *site_path.glob("vllm-*.dist-info"),
            site_path / "torch",
            *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision",
            *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio",
            *site_path.glob("torchaudio-*.dist-info"),
        ]:
            if leftover.exists():
                print("Removing leftover:", leftover)
                shutil.rmtree(leftover) if leftover.is_dir() else leftover.unlink()


    # Force clear GPU memory and reload torch
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except:
        pass

    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv", "pip", "install",
        "--system",
        VLLM_SPEC,
        "torch", "torchvision", "torchaudio",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

scipy_check = run_capture([
    sys.executable,
    "-c",
    "import numpy, scipy; print('numpy', numpy.__version__); print('scipy', scipy.__version__)",
])
if scipy_check.returncode:
    print(scipy_check.stdout)
    print(scipy_check.stderr)
    run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy", "scipy"])

run([
    sys.executable,
    "-c",
    "import torch, vllm; print('torch', torch.__version__); print('vllm', getattr(vllm, '__version__', 'unknown'))",
])

# run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])
# FIXED (Skip pip install, just add path)
# Only run pip install if setup.py is necessary for dependencies
print("Skipping pip install -e to avoid compilation/OOM risks. Adding to sys.path directly...")
plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

#Verify plugin loads correctly without pip install -e
try:
    # Only attempt to import if the plugin doesn't require compilation
    import importlib.util
    spec = importlib.util.spec_from_file_location("vllm_hook_plugins", PKG_DIR / "__init__.py")
    if spec:
        importlib.util.module_from_spec(spec)
        print("Plugin module loaded successfully from sys.path.")
except Exception as e:
    print(f"Warning: Plugin load from sys.path failed ({e}). Attempting pip install -e...")
    try:
        run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])
    except Exception as e:
        raise RuntimeError(f"Plugin installation failed. Ensure CUDA/C++ compilers are available. Error: {e}")


print("Plugin source:", plugin_src_dir)
print("Python exec  :", sys.executable)

if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("Restarting Colab runtime once so Python reloads the replaced vLLM/Torch binary packages.")
    # Add a small sleep to allow output buffer to flush before killing
    import time
    time.sleep(1)
    os.kill(os.getpid(), 9)

subprocess_smoke_check()


Colab detected. Reusing existing clone at /content/vLLM-Hook
Refreshing existing clone ...
Running: git -C /content/vLLM-Hook fetch origin main
From https://github.com/IBM/vLLM-Hook
 * branch            main       -> FETCH_HEAD
Running: git -C /content/vLLM-Hook checkout main
Already on 'main'
Your branch is up to date with 'origin/main'.
Running: git -C /content/vLLM-Hook pull --ff-only origin main
From https://github.com/IBM/vLLM-Hook
 * branch            main       -> FETCH_HEAD
Already up to date.
Changed working directory to /content/vLLM-Hook/notebooks
Colab      : True
Notebook dir: /content/vLLM-Hook/notebooks
Repo root   : /content/vLLM-Hook
Repo branch : main
Package dir : /content/vLLM-Hook/vllm_hook_plugins
Req file    : /content/vLLM-Hook/requirement.txt
Skipping requirement managed by Colab install cell: vllm>=0.5
Skipping requirement managed by Colab install cell: torch>=2.0
Skipping requirement managed by Colab install cell: numpy>=1.24
Running: /usr/bin/python3 -m pip 

### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like vllm.LLM (`from vllm import LLM`) but adds support for hooks and instrumentation.
We import it here:

In [2]:
from vllm_hook_plugins import HookLLM

### Environment & multiprocessing setup

In [3]:
import os
import multiprocessing as mp
import torch
from pathlib import Path
from vllm import SamplingParams
mp.set_start_method("spawn", force=True)
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Steering vector paths in model_configs/.../*.json are relative to the repo
# root, so chdir there before constructing HookLLM.
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

### Initialize `HookLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [4]:
cache_dir = './cache'  # Specify cache dir
model = 'microsoft/Phi-3-mini-4k-instruct'

dtype_map = {
    'microsoft/Phi-3-mini-4k-instruct': 'auto',
}

We also need to provide a config file that specifies how activations are steered (e.g., which layers to intervene on, which token to intervene, what direction vectors to apply, etc.).<br />
In the following example, we apply activation steering at the 15th layer, apply the steering at all positions (as opposed to only at the start of the decoding process), and along the direction given in `vector_path`:

In [5]:
import json

json_path = Path("model_configs/activation_steer/Phi-3-mini-4k-instruct.json")  # adjust path

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)

Inside `steer_hook_act` we defined the activation steering behavior during model inference.
Now, we initialize the llm:

In [6]:
llm = HookLLM(
    model=model,
    worker_name="steer_hook_act",
    config_file=json_path,
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_prefix_caching=True,
    enable_hook=True,
)

INFO 06-02 01:43:31 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': './cache', 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'worker_extension_cls': 'vllm_hook_plugins.workers.steer_activation_worker.SteerHookActWorker', 'model': 'microsoft/Phi-3-mini-4k-instruct'}
WARNING 06-02 01:43:31 [envs.py:1717] Unknown vLLM environment variable detected: VLLM_USE_V1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

INFO 06-02 01:47:11 [llm.py:391] Supported tasks: ['generate']


### Test case
In the following, we show a test case and compare generations **with** and **without** activation steering.

**Note**: Users should swap the example configs with their own to show desirable performance. The following is for pipeline illustration only.

In [7]:
test_cases = [
    "Write a dialogue between two people, one is dressed up in a ball gown and the other is dressed down in sweats. The two are going to a nightly event. Your answer must contain exactly 3 bullet points in the markdown format (use \"* \" to indicate each bullet) such as:\n* This is the first point.\n* This is the second point.",
    "What is the difference between the 13 colonies and the other British colonies in North America? Your answer must contain exactly 6 bullet point in Markdown using the following format:\n* Bullet point one.\n* Bullet point two.\n...\n* Bullet point fix."
]

Before we start, we define the sampling parameters:

**Note**: token 32007 is phi-specific, refer to the original huggingface implementation for details https://github.com/microsoft/llm-steer-instruct/blob/main/utils/generation_utils.py.

- Option 1: we can use a uniform sampling parameters as follows

In [8]:
# sampling_params = SamplingParams(
#     temperature=0.0,
#     max_tokens=2048,
#     stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
# )

- Option 2: we can change the per-request steering strength by adding `extra_args`. For example, below, we apply a stronger steering coefficient and different steering method to the second prompt than the first, all other steer settings unchanged.

In [9]:
sampling_params_list = [
    SamplingParams(
        temperature=0.0,
        max_tokens=2048,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
    ),
    SamplingParams(
        temperature=0.0,
        max_tokens=2048,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
        extra_args={"steer": {"method":"add_vector", "coefficient": 10}},
    ),
]

Next, for each prompt, we:
1. Apply chat template on the test cases
2. Generate with activation steering enabled (`use_hook=True`, default),
3. Reset the prefix cache to ensure the baseline generation does not reuse steered cache,
4. Generate again with `use_hook=False` to obtain the baseline output.

In [10]:
outputs = []
outputs_original = []

for case, sampling_params in zip(test_cases, sampling_params_list):
    print("=" * 100)
    prompt = case
    messages = [{"role": "user", "content": prompt}]
    example = llm.tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    outputs.extend(llm.generate(example, sampling_params))

    llm.llm_engine.reset_prefix_cache()

    outputs_original.extend(llm.generate(example, sampling_params, use_hook=False))

    llm.llm_engine.reset_prefix_cache()

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Finally we can print out the results as follows:

In [11]:
for steered, original in zip(outputs, outputs_original):
    print("=" * 100)
    steered_text = steered.outputs[0].text
    print("\n[With activation steering]\n")
    print(steered_text)

    baseline_text = original.outputs[0].text
    print("\n[Without activation steering]\n")
    print(baseline_text)


[With activation steering]

 * The woman in the ball gown is excitedly discussing the upcoming event with her friend, who is dressed in casual sweats.
* The friend, feeling out of place in their sweats, expresses their discomfort and asks if they can borrow a dress from the woman.
* The woman, understanding her friend's predicament, offers to lend them a simple yet elegant dress that will make them feel more comfortable and confident at the event.

[Without activation steering]

 * The woman in the ball gown is excitedly discussing the upcoming event with her friend, who is dressed in casual sweats.
* The friend, feeling out of place, expresses their discomfort with the formal attire and suggests they skip the event altogether.
* The woman in the ball gown reassures her friend that they will have a great time and offers to help them find a more comfortable outfit to wear.

[With activation steering]

 * The 13 colonies were the first British colonies established in North America, whil